In [ ]:
import pandas as pd
from ipynb.fs.full.Database import get_connection  # type: ignore

In [ ]:
def consultar_datos():
    conn = get_connection()
    if conn:
        try:
            cursor = conn.cursor()
            # Ejemplo de consulta simple
            cursor.execute("SELECT T0.ItemCode AS 'Producto', T0.ItemName AS 'Descripción', T0.FirmCode AS 'Fabricante', T0.AvgPrice AS 'Costo', SUM(T1.OnHand) AS 'Stock Total' FROM OITM T0 INNER JOIN OITW T1 ON T0.ItemCode = T1.ItemCode WHERE T0.FirmCode = '39' GROUP BY T0.ItemCode, T0.ItemName, T0.FirmCode,T0.AvgPrice;")
            
            # Recuperar los nombres de las columnas
            columns = [column[0] for column in cursor.description]
            
            # Crear una lista de diccionarios para que sea fácil de leer
            resultados = []
            for row in cursor.fetchall():
                resultados.append(dict(zip(columns, row)))
            
            return resultados
        except Exception as e:
            print(f"Error en la consulta: {e}")
        finally:
            conn.close() # Siempre cierra la conexión
    return []

In [ ]:
""" def buscar_por_id(id_busqueda):
    conn = get_connection()
    if conn:
        try:
            cursor = conn.cursor()
            # Usamos parámetros (?) para evitar Inyección SQL
            query = "SELECT ItemCode, ItemName, AvgPrice, OnHand FROM OITM WHERE FirmCode IN (SELECT FirmCode FROM OMRC WHERE FirmName LIKE '%Siemens%')'"
            cursor.execute(query, (id_busqueda,))
            
            row = cursor.fetchone()
            return row
        finally:
            conn.close() """

' def buscar_por_id(id_busqueda):\n    conn = get_connection()\n    if conn:\n        try:\n            cursor = conn.cursor()\n            # Usamos parámetros (?) para evitar Inyección SQL\n            query = "SELECT ItemCode, ItemName, AvgPrice, OnHand FROM OITM WHERE FirmCode IN (SELECT FirmCode FROM OMRC WHERE FirmName LIKE \'%Siemens%\')\'"\n            cursor.execute(query, (id_busqueda,))\n            \n            row = cursor.fetchone()\n            return row\n        finally:\n            conn.close() '

In [ ]:
def exportar_usuarios_a_csv(nombre_archivo="datos_SQL.csv"):
    conn = get_connection()
    if conn:
        try:
            query = "SELECT T0.ItemCode AS 'Producto', T0.ItemName AS 'Descripción', T0.FirmCode AS 'Fabricante', T0.AvgPrice AS 'Costo', SUM(T1.OnHand) AS 'Stock Total' FROM OITM T0 INNER JOIN OITW T1 ON T0.ItemCode = T1.ItemCode WHERE T0.FirmCode = '39' GROUP BY T0.ItemCode, T0.ItemName, T0.FirmCode,T0.AvgPrice;"
            
            # 1. Crear el DataFrame directamente desde el query
            # Pandas lee la conexión y ejecuta el SQL por ti
            df = pd.read_sql(query, conn)
            
            # 2. Transformar el DataFrame en un CSV
            # index=False evita que se guarde una columna extra con los números de fila
            df.to_csv(nombre_archivo, index=False, encoding='utf-8-sig', sep=';')
            
            print(f"Éxito: Archivo '{nombre_archivo}' creado con {len(df)} registros.")
            return df
            
        except Exception as e:
            print(f"Error al procesar datos o crear CSV: {e}")
        finally:
            conn.close()
    return None